In [ ]:
import os
import sys
import spacy
import importlib
import config
importlib.reload(config)

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

data_path_kafka = os.path.join(module_path, r"data\kafka_korpus")



def read_txt_file(file_path):
    """
    Read a single txt file and return its content as a string.

    """
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

In [14]:
def read_files_from_folders(folders: list[str]) -> list[dict]:
    from os import listdir
    from os.path import isfile, join

    files = {}
    for folder in folders:

        for f in listdir(folder):
            if isfile(join(folder, f)):
                name = os.path.splitext(f)[0]
                files[name] = os.path.join(folder, f)
        
    return files


In [15]:
nlp = spacy.load(config.SPACY_MODEL_DE)

das_schloss = read_txt_file(os.path.join(data_path_kafka, r"roman_fragmente\das_schloss.txt"))

doc = nlp(das_schloss)

### Wörterbuch-basierte Validierung für un-Negationen

**Problem**: Der bisherige Ansatz prüfte, ob der "Rest" (z.B. *ruhig* aus *unruhig*) im selben Text vorkommt oder von spaCy als bekannt erkannt wird. Beides versagt:
- **Korpus-Check**: Wenn Kafka nur *unruhig* schreibt, aber nie *ruhig*, fällt das Wort durch
- **`is_oov` mit `sm`-Modell**: Das kleine spaCy-Modell hat keine Wortvektoren → `is_oov` ist fast immer `True`

**Lösung**: Kombination aus:
1. **`wordfreq`** – Frequenzwörterbuch mit ~400k deutschen Einträgen → "Ist *ruhig* ein bekanntes deutsches Wort?"
2. **Morphologische Suffix-Heuristik** – Endet der Rest auf typische deutsche Suffixe (-lich, -ig, -bar, -heit, etc.)?
3. **Korpus-Präsenz** bleibt als Zusatzinfo erhalten (nicht mehr als Filter)

In [16]:
from wordfreq import word_frequency

# --- Deutsche morphologische Suffixmuster ---
# Typische Endungen, die auf valide Adjektive / Nomen / Adverbien hindeuten
ADJEKTIV_SUFFIXE = {
    "lich", "ig", "bar", "sam", "haft", "los", 
    "reich", "voll", "arm", "fest", "frei", "mäßig"
}
NOMEN_SUFFIXE = {
    "heit", "keit", "ung", "schaft", "tum", "nis", 
    "sal", "mut", "sinn", "kraft", "lust"
}
MORPHO_SUFFIXE = ADJEKTIV_SUFFIXE | NOMEN_SUFFIXE


def ist_bekanntes_deutsches_wort(wort: str, schwelle: float = 0) -> dict:
    """
    Prüft ob ein Wort ein bekanntes deutsches Wort ist.
    
    Schicht 1: wordfreq – Frequenzwörterbuch (sehr zuverlässig)
    Schicht 2: Morphologische Suffix-Heuristik (Fallback)
    
    Args:
        wort: Das zu prüfende Wort
        schwelle: Minimale Frequenz (default 0 = jede bekannte Frequenz reicht)
    
    Returns:
        dict mit 'bekannt' (bool), 'quelle' (str), 'frequenz' (float)
    """
    wort_lower = wort.lower()
    freq = word_frequency(wort_lower, 'de')
    
    # Schicht 1: wordfreq kennt das Wort
    if freq > schwelle:
        return {"bekannt": True, "quelle": "wordfreq", "frequenz": freq}
    
    # Schicht 2: Morphologische Suffix-Heuristik
    # Wenn der Rest auf ein typisches deutsches Suffix endet, ist es wahrscheinlich ein valides Wort
    for suffix in MORPHO_SUFFIXE:
        if wort_lower.endswith(suffix) and len(wort_lower) > len(suffix) + 2:
            return {"bekannt": True, "quelle": "suffix_heuristik", "frequenz": freq}
    
    return {"bekannt": False, "quelle": "keine", "frequenz": freq}


# --- Schnelltest ---
testwoerter = ["ruhig", "geduld", "möglich", "glück", "bekannt", "sinn", "freundlich", "ordnung", "xyzfoo"]
for w in testwoerter:
    r = ist_bekanntes_deutsches_wort(w)
    print(f"  {w:20s} → bekannt: {r['bekannt']}, quelle: {r['quelle']}, freq: {r['frequenz']:.2e}")

  ruhig                → bekannt: True, quelle: wordfreq, freq: 5.75e-05
  geduld               → bekannt: True, quelle: wordfreq, freq: 1.78e-05
  möglich              → bekannt: True, quelle: wordfreq, freq: 2.51e-04
  glück                → bekannt: True, quelle: wordfreq, freq: 1.74e-04
  bekannt              → bekannt: True, quelle: wordfreq, freq: 1.82e-04
  sinn                 → bekannt: True, quelle: wordfreq, freq: 1.07e-04
  freundlich           → bekannt: True, quelle: wordfreq, freq: 2.75e-05
  ordnung              → bekannt: True, quelle: wordfreq, freq: 8.51e-05
  xyzfoo               → bekannt: False, quelle: keine, freq: 0.00e+00


In [17]:
def find_un_negations(doc, nlp) -> list[dict]:
    '''
    Findet mit "un-" negierte Wörter in einem Text.

    Validierung des Rests (ohne "un-") über:
      1. wordfreq – deutsches Frequenzwörterbuch (~400k Einträge)
      2. Morphologische Suffix-Heuristik (Fallback)
      3. Korpus-Lookup nur als Zusatzinfo, NICHT als Filter

    Args:
        doc: spaCy Doc-Objekt
        nlp: spaCy Language-Objekt

    Returns:
        Liste von Dicts mit Infos zu den gefundenen Negationen
    '''
    corpus_lemmas = {token.lemma_.lower() for token in doc if token.is_alpha}
    relevant_pos_tags = {"ADJ", "NOUN", "ADV", "VERB"}

    # Erweiterte Stoppliste: Wörter mit "un-" die KEINE Negation sind
    false_positives = {
        # Grundformen
        "und", "uns", "unser", "unter", "ungefähr", "ungeheuer",
        "um", "ungarn", "universal", "universität", "uniform",
        "union", "unitarisch", "unikat", "unison", "ungar",
        "unto", "unique", "unit", "universe",
        # "unter-" Komposita (kein un-Präfix!)
        "unternehmen", "unterbrechen", "unterscheiden", "unterstützen",
        "untersuchen", "unterschied", "unterricht", "unterhaltung",
        "unterhalb", "unterbrechung", "untergang", "untergehen",
        "unterhalten", "unterlassen", "untertauchen", "unterwerfen",
        "unterziehn", "unterziehen",
        # "unser-" Deklinationsformen
        "unserig", "unsern", "unsere", "unserem", "unseren", "unserer",
        "unseres", "unsrig",
    }

    results = []
    seen_lemmas = set()

    for token in doc:
        if not token.is_alpha:
            continue
        
        lemma = token.lemma_.lower()
        
        # Bereits gesehen?
        if lemma in seen_lemmas:
            continue
        
        # Beginnt mit "un" und ist lang genug?
        if not lemma.startswith("un") or len(lemma) <= 3:
            continue
        
        # POS-Tag relevant?
        if token.pos_ not in relevant_pos_tags:
            continue
    
        # In der Stoppliste?
        if lemma in false_positives:
            continue
        
        # "unter-" Präfix erkennen (auch wenn nicht explizit in Stoppliste)
        if lemma.startswith("unter"):
            continue

        # Rest nach "un" extrahieren
        rest = lemma[2:]
        
        # --- Validierung: Wörterbuch + Heuristik ---
        pruefung = ist_bekanntes_deutsches_wort(rest)
        im_korpus = rest in corpus_lemmas
        
        if pruefung["bekannt"]:
            seen_lemmas.add(lemma)
            results.append({
                "lemma": lemma,
                "rest": rest,
                "pos": token.pos_,
                "beispiel": token.text,
                "im_korpus": im_korpus,
                "validierung": pruefung["quelle"],
                "rest_frequenz": pruefung["frequenz"]
            })
    
    return results

In [18]:
josefine = read_txt_file(os.path.join(data_path_kafka, r"erzählungen\josefine_die_saengerin.txt"))

doc = nlp(josefine)
un_negations = find_un_negations(doc, nlp)

print(f"Gefundene un-Negationen: {len(un_negations)}\n")
for item in sorted(un_negations, key=lambda x: x["lemma"]):
    korpus_marker = "✓" if item["im_korpus"] else "✗"
    print(f"  {item['lemma']:25s} → {item['rest']:20s} (POS: {item['pos']}, via: {item['validierung']:15s}, im Korpus: {korpus_marker})")

Gefundene un-Negationen: 37

  unausrottbar              → ausrottbar           (POS: ADJ, via: suffix_heuristik, im Korpus: ✗)
  unbeachtet                → beachtet             (POS: ADV, via: wordfreq       , im Korpus: ✗)
  unbefangen                → befangen             (POS: ADJ, via: wordfreq       , im Korpus: ✗)
  unbegreiflich             → begreiflich          (POS: ADJ, via: wordfreq       , im Korpus: ✗)
  unbegründet               → begründet            (POS: ADV, via: wordfreq       , im Korpus: ✗)
  unbeherrscht              → beherrscht           (POS: ADJ, via: wordfreq       , im Korpus: ✗)
  unbekannt                 → bekannt              (POS: ADV, via: wordfreq       , im Korpus: ✗)
  unberechenbar             → berechenbar          (POS: ADV, via: wordfreq       , im Korpus: ✗)
  unbeschränken             → beschränken          (POS: VERB, via: wordfreq       , im Korpus: ✗)
  undeutbar                 → deutbar              (POS: ADJ, via: wordfreq       , im 

In [19]:
def compute_un_negation_metrics(un_negations: list, doc) -> dict:
    '''
    Berechnet Metriken aus den gefundenen un-Negationen.

    Args:
        un_negations: Liste von Dicts aus find_un_negations()
        doc: spaCy Doc-Objekt

    Returns:
        Dict mit berechneten Metriken
    '''
    from collections import Counter

    total_tokens = len([t for t in doc if t.is_alpha])
    sentences = list(doc.sents)
    un_lemmas = {item["lemma"] for item in un_negations}

    # --- 1. Häufigkeitsmetriken ---
    # Alle Vorkommen zählen (nicht nur unique)
    all_un_occurrences = [
        token for token in doc 
        if token.is_alpha and token.lemma_.lower() in un_lemmas
    ]
    un_token_count = len(all_un_occurrences)
    un_type_count = len(un_negations)  # unique Lemmas
    density_per_1000 = (un_token_count / total_tokens) * 1000 if total_tokens > 0 else 0
    ttr = un_type_count / un_token_count if un_token_count > 0 else 0

    # --- 2. Verteilung pro Satz ---
    sents_with_un = []
    for sent in sentences:
        count = sum(
            1 for token in sent 
            if token.is_alpha and token.lemma_.lower() in un_lemmas
        )
        sents_with_un.append(count)

    sents_with_at_least_one = sum(1 for c in sents_with_un if c > 0)
    avg_per_sent = sum(sents_with_un) / len(sentences) if sentences else 0
    max_per_sent = max(sents_with_un) if sents_with_un else 0

    # --- 3. POS-Verteilung ---
    pos_distribution = Counter(item["pos"] for item in un_negations)

    # --- 4. Un-Ratio: un-Form vs. Positivform ---
    un_vs_positive = {}
    for item in un_negations:
        lemma = item["lemma"]
        rest = item["rest"]
        un_count = sum(1 for t in doc if t.is_alpha and t.lemma_.lower() == lemma)
        pos_count = sum(1 for t in doc if t.is_alpha and t.lemma_.lower() == rest)
        un_vs_positive[lemma] = {
            "un_count": un_count,
            "positive_count": pos_count,
            "ratio": un_count / pos_count if pos_count > 0 else float("inf")
        }

    return {
        "total_tokens": total_tokens,
        "un_types": un_type_count,
        "un_tokens": un_token_count,
        "density_per_1000": round(density_per_1000, 2),
        "type_token_ratio": round(ttr, 3),
        "avg_per_sentence": round(avg_per_sent, 4),
        "max_per_sentence": max_per_sent,
        "sentences_total": len(sentences),
        "sentences_with_un": sents_with_at_least_one,
        "sentences_with_un_pct": round(sents_with_at_least_one / len(sentences) * 100, 2) if sentences else 0,
        "pos_distribution": dict(pos_distribution),
        "un_vs_positive_ratio": un_vs_positive
    }

In [ ]:
import pandas as pd

# --- Alle Werke einlesen und analysieren ---
# Vollständige Pfade aus config Ordnern
folders = [os.path.join(data_path_kafka, name) for name in config.RELEVANTE_ORDNER]

werke = read_files_from_folders(folders)

results_list = []

for title, rel_path in werke.items():
    text = read_txt_file(os.path.join(data_path_kafka, rel_path))
    doc = nlp(text)
    un_neg = find_un_negations(doc, nlp)
    metrics = compute_un_negation_metrics(un_neg, doc)
    
    # Flache Zeile für den DataFrame bauen
    row = {
        "werk": title,
        "total_tokens": metrics["total_tokens"],
        "un_types": metrics["un_types"],
        "un_tokens": metrics["un_tokens"],
        "density_per_1000": metrics["density_per_1000"],
        "type_token_ratio": metrics["type_token_ratio"],
        "sentences_total": metrics["sentences_total"],
        "sentences_with_un": metrics["sentences_with_un"],
        "sentences_with_un_pct": metrics["sentences_with_un_pct"],
        "avg_per_sentence": metrics["avg_per_sentence"],
        "max_per_sentence": metrics["max_per_sentence"],
        # POS-Verteilung flachklopfen
        "pos_ADJ": metrics["pos_distribution"].get("ADJ", 0),
        "pos_NOUN": metrics["pos_distribution"].get("NOUN", 0),
        "pos_ADV": metrics["pos_distribution"].get("ADV", 0),
        "pos_VERB": metrics["pos_distribution"].get("VERB", 0)
    }
    results_list.append(row)

df_metrics = pd.DataFrame(results_list)
df_metrics = df_metrics.set_index("werk")
df_metrics

,total_tokens,un_types,un_tokens,density_per_1000,type_token_ratio,sentences_total,sentences_with_un,sentences_with_un_pct,avg_per_sentence,max_per_sentence,pos_ADJ,pos_NOUN,pos_ADV,pos_VERB
werk,,,,,,,,,,,,,,
amerika,83558,167,407,4.87,0.410,3965,371,9.36,0.1026,3,57,32,70,8
das_schloss,76013,188,443,5.83,0.424,3782,395,10.44,0.1171,3,46,49,87,6
der_process,72816,144,390,5.36,0.369,3869,363,9.38,0.1008,4,28,30,84,2
auf_der_galerie,288,1,1,3.47,1.000,15,1,6.67,0.0667,1,1,0,0,0
beim_bau_der_chinesischen_mauer,4291,26,28,6.53,0.929,201,26,12.94,0.1393,2,10,5,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
an_vater,16244,95,133,8.19,0.714,676,106,15.68,0.1967,4,22,25,44,4
reise_friedland_reichenberg,1804,11,14,7.76,0.786,110,12,10.91,0.1273,3,4,4,3,0
reise_lugano_paris_erlenbach,12102,44,63,5.21,0.698,892,56,6.28,0.0706,2,14,15,15,0


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df_metrics.sort_values(by='avg_per_sentence', ascending=False)

output_path = os.path.join(module_path, r"output\results_v1.csv")
df_metrics.to_csv(output_path, sep = ",", index = False)

# Vergleichskorpus

In [ ]:
import pandas as pd

# --- Alle Werke einlesen und analysieren ---
# Vollständige Pfade aus config Ordnern
folders = [os.path.join(module_path, r"data\Vergleichskorpus\corpus")]

werke = read_files_from_folders(folders)

results_list = []

for title, rel_path in werke.items():
    print(title)
    text = read_txt_file(os.path.join(data_path_kafka, rel_path))
    nlp.max_length = 1100000
    doc = nlp(text)
    
    un_neg = find_un_negations(doc, nlp)
    metrics = compute_un_negation_metrics(un_neg, doc)
    
    # Flache Zeile für den DataFrame bauen
    row = {
        "werk": title,
        "total_tokens": metrics["total_tokens"],
        "un_types": metrics["un_types"],
        "un_tokens": metrics["un_tokens"],
        "density_per_1000": metrics["density_per_1000"],
        "type_token_ratio": metrics["type_token_ratio"],
        "sentences_total": metrics["sentences_total"],
        "sentences_with_un": metrics["sentences_with_un"],
        "sentences_with_un_pct": metrics["sentences_with_un_pct"],
        "avg_per_sentence": metrics["avg_per_sentence"],
        "max_per_sentence": metrics["max_per_sentence"],
        # POS-Verteilung flachklopfen
        "pos_ADJ": metrics["pos_distribution"].get("ADJ", 0),
        "pos_NOUN": metrics["pos_distribution"].get("NOUN", 0),
        "pos_ADV": metrics["pos_distribution"].get("ADV", 0),
        "pos_VERB": metrics["pos_distribution"].get("VERB", 0)
    }
    results_list.append(row)

df_metrics_vergleichskorpus = pd.DataFrame(results_list)
df_metrics_vergleichskorpus = df_metrics_vergleichskorpus.set_index("werk")
df_metrics_vergleichskorpus

Beer-Hofmann_DerTodGeorgs
Beer-Hofmann_SchlafliedFu╠êrMirjam
Benn_Gehirne
Brod_ArnoldBeerSchicksalEinesJuden
Ehrenstein_DerMenschSchreit
Ehrenstein_Tubutsch
Ehrenstein_Zauberma╠êrchen
George_DerKrieg
George_DerSiebenteRing
George_DerSternDesBundes
George_DerTeppichDesLebens
Hauptmann_BahnwärterThiel
Hauptmann_DieWeber
Hauptmann_VorSonnenaufgang
Hesse_PeterCamenzind
Hesse_Siddhartha
Hesse_Steppenwolf
Hesse_UntermRad
Heym_DerGottDerStadt
Heym_DerKrieg
Heym_DieStadt
Heym_DieStädte
HMann_DerUntertan
HMann_ProfessorUnrat
Hofmannsthal_DerTorUndDerTod
Hofmannsthal_EinBrief
Hofmannsthal_TodDesTizian
Holz_Schlaf_FamilieSelicke
Holz_Schlaf_PapaHamlet
Kafka_DasSchlo├ƒ
Kafka_DasUrteil
Kafka_DerProzess
Kafka_DerVerschollene
Kafka_DieVerwandlung
Kafka_Hungerkünstler
Kafka_Strafkolonie
Lasker-Schu╠êler_Essays
Lasker-Schu╠êler_MeinBlauesKlavier
Lasker-Schu╠êler_Sphinx
Lasker-Schu╠êler_Weltende
Lasker-Schüler_DerPrinzVonTheben
Lasker-Schüler_EinAlterTibetteppich
Musil_DieVerwirrungeDesZo╠êglingsTo╠êrle

KeyError: "None of ['werk'] are in the columns"

un- präfixe finden
1. lemmas machen
2. "un" abtrennen 
3. Überprüfen, ob Rest des Wortes noch ein valides deutsches wort ist -> wenn nicht "un" - negiertes wort gefunden, wenn nicht, dann false positive


POS-Verteilung: Anteil ADJ vs. NOUN vs. ADV – negiert Kafka eher Eigenschaften (unfreundlich) oder Zustände (Unruhe)?

[E088] Text of length 1069315 exceeds maximum of 1000000. The parser and NER models require roughly 1GB of temporary memory per 100,000 characters in the input. This means long texts may cause memory allocation errors. If you're not using the parser or NER, it's probably safe to increase the `nlp.max_length` limit. The limit is in number of characters, so you can check whether your inputs are too long by checking `len(text)`.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

PLan: 
- vergleichskorpus mit in die analyse einbeziehen
- descriptive stats und ttest